# Lenguajes y Autómatas II (SCD-1016)
## Sesión 22 · Generación de TAC para Expresiones Aritmético-Lógicas

**Docente:** MCC. Iván Márquez Larios (`ijmarquezl`) · **Repositorio maestro:** https://github.com/ijmarquezl/tecnm-lenguajes-automatas-2

---

> **Pregunta guía:** ¿Cómo se traduce un árbol de expresión a una lista plana de instrucciones sin perder la precedencia?
>
> **Objetivo de la sesión:** Recorrer un AST en postorden y emitir una instrucción TAC por cada operador, devolviendo el nombre del temporal o de la hoja que contiene el resultado.

**Entregable:** tu traductor en artesano_sesion22_c.c, el harness en verde y la bitácora

**Tiempo:** 60 min · **Fases:** Artesano (manual) → Científico (IA con contrato) → Auditor (harness)

> ¿Dudas de C? Consulta la [guía de C](https://github.com/ijmarquezl/tecnm-lenguajes-automatas-2/blob/main/docs/guia-c.md) antes de empezar.


## 1. ¿Qué vas a construir?

Recibes la raíz de un árbol con hojas y operadores binarios. Debes escribir una línea TAC por operador en el arreglo lineas y devolver cuántas líneas emitiste. Los temporales ya están resueltos: úsalos con nuevo_temporal.

Al final de la sesión tendrás: **Un traductor en C que convierte un árbol binario en líneas TAC y respeta la precedencia con temporales _t0, _t1, ...**

Los conceptos y el porqué de esta sesión están en la presentación
([sesión 22](https://ijmarquezl.github.io/tecnm-lenguajes-automatas-2/presentaciones/sesion22.html)). Aquí solo está el trabajo: qué se
recibe, qué debe salir y cómo se comprueba.


## 2. El contrato exacto

El archivo `api_sesion22.h` declara la interfaz que el harness sabe probar.
Tu implementación **debe** respetar esas firmas; si las cambias, el harness no compila.

```c
#include "api_sesion22.h"

int generar_tac(const Nodo *raiz, char lineas[][64], int max);
```

- Primero se calculan los hijos y después se emite el operador: el recorrido es postorden.
- Cada operador usa un temporal nuevo: llama a nuevo_temporal, no inventes el nombre.
- Nunca escribas más de max líneas ni más de 63 caracteres por línea.

> El harness compila con `-Wall -Wextra -Werror`: una advertencia del compilador cuenta como error.
>
> El harness compila `harness_sesion22_c.c` junto con `artesano_sesion22_c.c`
> (o `ia_sesion22_c.c`). Por eso tu archivo **no debe tener `main()`**: el `main()`
> es del harness.


## 3. Cómo se califica

### Casos que evalúa el harness

| # | Caso | Resultado esperado |
| :-: | :--- | :--- |
| 1 | `a + b * c` | `_t0 = b * c` |
| 2 | `a + b * c (2a linea)` | `_t1 = a + _t0` |
| 3 | `(a + b) * c` | `_t1 = _t0 * c` |
| 4 | `x * (y - z) + w` | `_t2 = _t1 + w` |
| 5 | `hoja sin operaciones` | 0 |
| 6 | `(a + b) * (c - d)` | `_t2 = _t0 * _t1` |

### Salida literal que debes obtener

```text
=== AUDITORIA SESION 22 - Traductor de AST a TAC con precedencia ===
Caso                               esperado:              obtenido:              resultado
[ 1] a + b * c                          esperado: _t0 = b * c  obtenido: _t0 = b * c  OK
[ 2] a + b * c (2a linea)
     esperado: _t1 = a + _t0
     obtenido: _t1 = a + _t0
     -> OK
[ 3] (a + b) * c
     esperado: _t1 = _t0 * c
     obtenido: _t1 = _t0 * c
     -> OK
[ 4] x * (y - z) + w
     esperado: _t2 = _t1 + w
     obtenido: _t2 = _t1 + w
     -> OK
[ 5] hoja sin operaciones               esperado: 0            obtenido: 0            OK
[ 6] (a + b) * (c - d)
     esperado: _t2 = _t0 * _t1
     obtenido: _t2 = _t0 * _t1
     -> OK

RESULTADO: 6/6 pruebas superadas
TODO EN VERDE. Documenta tu auditoria en auditorias/auditoria_sesion22.md
```

Los números entre corchetes son el contador de casos. Mientras algo falle verás
`FALLA` en esa línea, el total será menor y el programa terminará con código de
salida 1.

### Criterios de aceptación

- [ ] Cada operador genera exactamente una línea y las hojas no generan ninguna.
- [ ] Los temporales se numeran en orden y no se reutilizan.
- [ ] El orden de las líneas respeta la precedencia del árbol.


## Paso 0 · Entorno


In [1]:
import os

if os.system('which gcc > /dev/null 2>&1') != 0:
    print('⏳ Instalando compilador...')
    !apt-get update -qq && apt-get install -y -qq gcc > /dev/null 2>&1

print('✅ Entorno listo:')
!gcc --version | head -n 1


✅ Entorno listo:
gcc (Ubuntu 13.3.0-6ubuntu2~24.04.1) 13.3.0


## Fase 1 · El Artesano (15 min, sin IA)

Escribe a mano la implementación en `artesano_sesion22_c.c`. El archivo ya trae la
firma y un `TODO`: edita el contenido de la celda `%%writefile` que está abajo (o el
archivo en el panel **Archivos** de Colab) y vuelve a ejecutarla para guardar los
cambios. Hazlo sin consultar a la IA. Si te atoras, anota en tu bitácora exactamente
en qué te atoraste; eso también es evidencia.


In [2]:
%%writefile api_sesion22.h
/* api_sesion22.h - contrato de la Sesion 22
 * Traductor de AST a TAC con precedencia
 *
 * Esta cabecera es el contrato: el harness y tu implementacion
 * deben incluirla. Si cambias una firma, el harness dejara de compilar.
 */
#ifndef API_SESION22_H
#define API_SESION22_H

typedef struct Nodo {
    char tipo;
    char op;
    char valor[16];
    struct Nodo *izq;
    struct Nodo *der;
} Nodo;

int generar_tac(const Nodo *raiz, char lineas[][64], int max);

#endif /* API_SESION22_H */


Writing api_sesion22.h


In [6]:
%%writefile artesano_sesion22_c.c
/* artesano_sesion22_c.c - Fase 1: El Artesano (sin IA)
 * Traductor de AST a TAC con precedencia.
 *
 * Implementa lo que declara api_sesion22.h.
 * El harness compila este archivo junto con el suyo: no escribas main().
 */
#include "api_sesion22.h"
#include <stdio.h>

/* Guarda los nombres de las variables temporales */
typedef struct {
    char nombres[64][16];
    int cantidad;
} Temporales;

/* Genera el nombre de una variable temporal */
static const char *crear_temporal(Temporales *temp, int *contador) {
    if (temp->cantidad >= 64) {
        return "_t?";
    }

    snprintf(temp->nombres[temp->cantidad], 16,
             "_t%d", *contador);

    (*contador)++;

    return temp->nombres[temp->cantidad++];
}

/* Recorre el arbol y genera las operaciones */
static const char *recorrer(const Nodo *n, char lineas[][64],
                            int max, int *cantidad,
                            Temporales *temp, int *contador) {
    if (n == NULL) {
        return "";
    }

    /* Si es una hoja, regresamos su valor */
    if (n->tipo == 'H') {
        return n->valor;
    }

    /* Primero revisamos los hijos */
    const char *izq = recorrer(n->izq, lineas, max,
                              cantidad, temp, contador);

    const char *der = recorrer(n->der, lineas, max,
                               cantidad, temp, contador);

    /* Generamos la operacion si hay espacio */
    if (*cantidad < max) {
        const char *nombre = crear_temporal(temp, contador);

        snprintf(lineas[*cantidad], 64,
                 "%s = %s %c %s", nombre, izq, n->op, der);

        *cantidad = *cantidad + 1;

        return nombre;
    }

    return "_t?";
}

/* Genera el codigo de tres direcciones */
int generar_tac(const Nodo *raiz, char lineas[][64], int max) {
    if (raiz == NULL || lineas == NULL || max <= 0) {
        return 0;
    }

    Temporales temp;
    temp.cantidad = 0;

    int contador = 0;
    int cantidad = 0;

    recorrer(raiz, lineas, max, &cantidad,
             &temp, &contador);

    return cantidad;
}

Writing artesano_sesion22_c.c


## Fase 2 · El Científico (10 min)

Redacta el contrato **C-R-E-O** (Contexto, Rol, Especificación, Output) para pedirle
la misma función a la IA, y pega el resultado en la celda de `ia_sesion22_c.c`. La
firma debe ser idéntica a la del contrato; si la IA la cambia, corrígela antes de
compilar.


### Contrato C-R-E-O sugerido para la IA

```markdown
**Contexto**:
Estoy trabajando en la Sesión 22 del curso de Lenguajes y Autómatas II, traduciendo un Árbol de Sintaxis Abstracta (AST) de expresiones aritmético-lógicas a Código de Tres Direcciones (TAC). Tengo un entorno en C y dispongo del archivo de cabecera `api_sesion22.h`.

**Rol**:
Actúa como un Ingeniero de Compiladores experto en C, especializado en generación de código intermedio y optimización de ASTs.

**Especificación**:
Debes implementar la función `int generar_tac(const Nodo *raiz, char lineas[][64], int max)` en C.
1. Realiza un recorrido en postorden (primero procesa el subárbol izquierdo, luego el derecho, y finalmente el nodo actual).
2. El nodo puede ser una hoja (tipo `'H'`) o un operador binario (tipo `'B'`).
3. Las hojas no emiten líneas TAC por sí solas, pero su valor o el temporal resultante debe retornarse para ser usado por sus padres.
4. Para cada operador binario (como `+`, `-`, `*`), genera una nueva línea de código de tres direcciones con el formato `"_tX = operando_izq op operando_der"` empleando un pool de temporales que se incremente de manera secuencial (`_t0`, `_t1`, etc.).
5. Copia cada instrucción TAC generada de forma segura en `lineas[n]` sin superar `max` líneas totales, asegurando no desbordar los 63 caracteres máximos por línea.
6. Retorna el número total de líneas TAC escritas.
7. La firma debe coincidir exactamente con la definida en `api_sesion22.h`. No incluyas una función `main`.

**Output**:
Genera únicamente el código C limpio, optimizado y libre de advertencias (`-Wall -Wextra -Werror`).
```

In [3]:
%%writefile ia_sesion22_c.c
/* ia_sesion22_c.c - Fase 2: El Cientifico
 *
 * Solución óptima generada que realiza recorrido en postorden
 * y genera la tabla de tres direcciones (TAC) de forma secuencial.
 */
#include "api_sesion22.h"
#include <stdio.h>
#include <string.h>

/* Pool para gestionar los temporales asignados de manera consecutiva */
typedef struct {
    char nombre[64][16];
    int usados;
} Pool;

static const char *nuevo_temporal(Pool *p, int *contador) {
    if (p->usados >= 64) return "_t?";
    snprintf(p->nombre[p->usados], sizeof p->nombre[0], "_t%d", (*contador)++);
    return p->nombre[p->usados++];
}

/* Función auxiliar recursiva para recorrer en postorden */
static const char *recorrer_ast(const Nodo *n, char lineas[][64], int max, int *lineas_emitidas, Pool *p, int *contador) {
    if (!n) return "";

    if (n->tipo == 'H') {
        return n->valor;
    }

    /* Recorrido Postorden: Primero hijos, luego padre */
    const char *izq_val = recorrer_ast(n->izq, lineas, max, lineas_emitidas, p, contador);
    const char *der_val = recorrer_ast(n->der, lineas, max, lineas_emitidas, p, contador);

    if (*lineas_emitidas < max) {
        const char *temp = nuevo_temporal(p, contador);
        snprintf(lineas[*lineas_emitidas], 64, "%s = %s %c %s", temp, izq_val, n->op, der_val);
        (*lineas_emitidas)++;
        return temp;
    }

    return "_t?";
}

int generar_tac(const Nodo *raiz, char lineas[][64], int max) {
    if (!raiz) return 0;

    Pool p;
    p.usados = 0;
    int contador = 0;
    int lineas_emitidas = 0;

    recorrer_ast(raiz, lineas, max, &lineas_emitidas, &p, &contador);
    return lineas_emitidas;
}


Writing ia_sesion22_c.c


## Fase 3 · El Auditor (10 min)

El harness de abajo **no aprueba nada por sí solo**: ejecuta tu función caso por caso,
la compara contra el valor esperado e imprime `OK` o `FALLA`. Léelo antes de ejecutarlo:
los casos te dicen qué se está midiendo.


In [4]:
%%writefile harness_sesion22_c.c
/* harness_sesion22_c.c - auditoria automatica de la Sesion 22
 *
 * Compilacion:
 *   gcc -std=c11 -Wall -Wextra -Werror harness_sesion22_c.c artesano_sesion22_c.c -o test_artesano
 *
 * El harness NO aprueba nada por si solo: cada linea se calcula
 * ejecutando tu implementacion y comparandola contra el valor esperado.
 * Si algun caso falla, el programa termina con codigo de salida 1.
 */
#include <stdio.h>
#include <string.h>
#include "api_sesion22.h"

static int probadas = 0;
static int fallidas = 0;

static void verificar_str(const char *caso, const char *esperado,
                          const char *obtenido) {
    int ok = (obtenido != NULL) && (strcmp(esperado, obtenido) == 0);
    probadas++;
    if (!ok) fallidas++;
    printf("[%2d] %-34s esperado: %-12s obtenido: %-12s %s\n",
           probadas, caso, esperado, obtenido ? obtenido : "(NULL)",
           ok ? "OK" : "FALLA");
}

static void verificar_str_largo(const char *caso, const char *esperado,
                                const char *obtenido) {
    int ok = (obtenido != NULL) && (strcmp(esperado, obtenido) == 0);
    probadas++;
    if (!ok) fallidas++;
    printf("[%2d] %s\n", probadas, caso);
    printf("     esperado: %s\n", esperado);
    printf("     obtenido: %s\n", obtenido ? obtenido : "(NULL)");
    printf("     -> %s\n", ok ? "OK" : "FALLA");
}

static void verificar_int(const char *caso, long esperado, long obtenido) {
    int ok = (esperado == obtenido);
    probadas++;
    if (!ok) fallidas++;
    printf("[%2d] %-34s esperado: %-12ld obtenido: %-12ld %s\n",
           probadas, caso, esperado, obtenido, ok ? "OK" : "FALLA");
}

int main(void) {
    printf("=== AUDITORIA SESION 22 - Traductor de AST a TAC con precedencia ===\n");
    printf("Caso                               esperado:              obtenido:              resultado\n");

    { Nodo a = {.tipo='H', .valor="a"}; Nodo b = {.tipo='H', .valor="b"}; Nodo c = {.tipo='H', .valor="c"}; Nodo m = {.tipo='B', .op='*', .izq=&b, .der=&c}; Nodo s = {.tipo='B', .op='+', .izq=&a, .der=&m}; char l[8][64] = {{0}}; generar_tac(&s, l, 8); verificar_str("a + b * c", "_t0 = b * c", l[0]); }
    { Nodo a = {.tipo='H', .valor="a"}; Nodo b = {.tipo='H', .valor="b"}; Nodo c = {.tipo='H', .valor="c"}; Nodo m = {.tipo='B', .op='*', .izq=&b, .der=&c}; Nodo s = {.tipo='B', .op='+', .izq=&a, .der=&m}; char l[8][64] = {{0}}; generar_tac(&s, l, 8); verificar_str_largo("a + b * c (2a linea)", "_t1 = a + _t0", l[1]); }
    { Nodo a = {.tipo='H', .valor="a"}; Nodo b = {.tipo='H', .valor="b"}; Nodo c = {.tipo='H', .valor="c"}; Nodo s = {.tipo='B', .op='+', .izq=&a, .der=&b}; Nodo m = {.tipo='B', .op='*', .izq=&s, .der=&c}; char l[8][64] = {{0}}; generar_tac(&m, l, 8); verificar_str_largo("(a + b) * c", "_t1 = _t0 * c", l[1]); }
    { Nodo x = {.tipo='H', .valor="x"}; Nodo y = {.tipo='H', .valor="y"}; Nodo z = {.tipo='H', .valor="z"}; Nodo w = {.tipo='H', .valor="w"}; Nodo r = {.tipo='B', .op='-', .izq=&y, .der=&z}; Nodo m = {.tipo='B', .op='*', .izq=&x, .der=&r}; Nodo s = {.tipo='B', .op='+', .izq=&m, .der=&w}; char l[8][64] = {{0}}; generar_tac(&s, l, 8); verificar_str_largo("x * (y - z) + w", "_t2 = _t1 + w", l[2]); }
    { Nodo a = {.tipo='H', .valor="x"}; char l[4][64] = {{0}}; int n = generar_tac(&a, l, 4); verificar_int("hoja sin operaciones", (long)(0), (long)(n)); }
    { Nodo a = {.tipo='H', .valor="a"}; Nodo b = {.tipo='H', .valor="b"}; Nodo c = {.tipo='H', .valor="c"}; Nodo d = {.tipo='H', .valor="d"}; Nodo s1 = {.tipo='B', .op='+', .izq=&a, .der=&b}; Nodo s2 = {.tipo='B', .op='-', .izq=&c, .der=&d}; Nodo m = {.tipo='B', .op='*', .izq=&s1, .der=&s2}; char l[8][64] = {{0}}; generar_tac(&m, l, 8); verificar_str_largo("(a + b) * (c - d)", "_t2 = _t0 * _t1", l[2]); }

    printf("\nRESULTADO: %d/%d pruebas superadas\n",
           probadas - fallidas, probadas);
    if (fallidas != 0) {
        printf("Aun hay %d caso(s) sin resolver. Revisa tu implementacion.\n",
               fallidas);
        return 1;
    }
    printf("TODO EN VERDE. Documenta tu auditoria en auditorias/auditoria_sesion22.md" "\n");
    return 0;
}


Writing harness_sesion22_c.c


In [7]:
!gcc -std=c11 -Wall -Wextra -Werror harness_sesion22_c.c artesano_sesion22_c.c -o test_artesano && ./test_artesano

=== AUDITORIA SESION 22 - Traductor de AST a TAC con precedencia ===
Caso                               esperado:              obtenido:              resultado
[ 1] a + b * c                          esperado: _t0 = b * c  obtenido: _t0 = b * c  OK
[ 2] a + b * c (2a linea)
     esperado: _t1 = a + _t0
     obtenido: _t1 = a + _t0
     -> OK
[ 3] (a + b) * c
     esperado: _t1 = _t0 * c
     obtenido: _t1 = _t0 * c
     -> OK
[ 4] x * (y - z) + w
     esperado: _t2 = _t1 + w
     obtenido: _t2 = _t1 + w
     -> OK
[ 5] hoja sin operaciones               esperado: 0            obtenido: 0            OK
[ 6] (a + b) * (c - d)
     esperado: _t2 = _t0 * _t1
     obtenido: _t2 = _t0 * _t1
     -> OK

RESULTADO: 6/6 pruebas superadas
TODO EN VERDE. Documenta tu auditoria en auditorias/auditoria_sesion22.md


In [5]:
!gcc -std=c11 -Wall -Wextra -Werror harness_sesion22_c.c ia_sesion22_c.c -o test_ia && ./test_ia

=== AUDITORIA SESION 22 - Traductor de AST a TAC con precedencia ===
Caso                               esperado:              obtenido:              resultado
[ 1] a + b * c                          esperado: _t0 = b * c  obtenido: _t0 = b * c  OK
[ 2] a + b * c (2a linea)
     esperado: _t1 = a + _t0
     obtenido: _t1 = a + _t0
     -> OK
[ 3] (a + b) * c
     esperado: _t1 = _t0 * c
     obtenido: _t1 = _t0 * c
     -> OK
[ 4] x * (y - z) + w
     esperado: _t2 = _t1 + w
     obtenido: _t2 = _t1 + w
     -> OK
[ 5] hoja sin operaciones               esperado: 0            obtenido: 0            OK
[ 6] (a + b) * (c - d)
     esperado: _t2 = _t0 * _t1
     obtenido: _t2 = _t0 * _t1
     -> OK

RESULTADO: 6/6 pruebas superadas
TODO EN VERDE. Documenta tu auditoria en auditorias/auditoria_sesion22.md


## Resultado esperado

Si tu implementación es correcta, la última línea del harness será:

```text
RESULTADO: 6/6 pruebas superadas
```

y la celda terminará sin error (código de salida 0). Compara tu salida completa
contra la de la sección 3: debe ser idéntica, línea por línea.


## Entrega

1. Copia `plantillas/auditoria_sesion22.md` a `auditorias/auditoria_sesion22.md` en tu fork.
2. Adjunta la salida del harness (las 6 líneas con `OK`).
3. Explica en el reporte crítico qué caso borde omitió la IA y cómo lo detectaste.
4. Haz commit y sincroniza tu fork.

Este laboratorio alimenta el 40% de laboratorios y es la base del Hito correspondiente
del Proyecto Integrador.
